In [ ]:
print(superstore_df[superstore_df["Order ID"].isin(multi_customer_orders.index)][
    ["Order ID", "Customer ID", "Customer Name", "Product ID", "Product Name"]
])
customer_ids = ["HO-15231", "HO-15232", "HO-15233", "HO-15234"]

print(superstore_df[superstore_df["Customer ID"].isin(customer_ids)][
    ["Customer ID", "Customer Name"]
].drop_duplicates().sort_values("Customer ID"))

print(superstore_df[superstore_df["Customer ID"].isin(["HO-15231", "HO-15232", "HO-15233", "HO-15234"])][
    ["Customer ID", "Customer Name", "Order ID", "Sales", "Profit"]
].sort_values(["Customer ID", "Order ID"]))

print(superstore_df[superstore_df["Order ID"].isin(["CA-2025-121465", "CA-2026-130494"])][
    ["Order ID", "Order Date", "Ship Date", "Product ID", "Product Name",
     "Sales", "Quantity", "Discount", "Profit"]
].sort_values(["Order ID", "Product ID"]))

### Repeated Order Composition

Two separate orders contain the same four products with identical quantities, discounts, sales, and profits.

Because the orders occurred on different dates, this is treated as a notable pattern rather than a confirmed data-quality error.

In [ ]:
print(superstore_df["Product ID"].nunique())
print(superstore_df["Order ID"].nunique())
print(superstore_df["Customer ID"].nunique())
print(len(superstore_df))

print(superstore_df.groupby("Order ID")["Customer ID"].nunique().value_counts())

multi_customer_orders = (
    superstore_df.groupby("Order ID")["Customer ID"]
      .nunique()
      .loc[lambda x: x > 1]
)

print(multi_customer_orders)

print(superstore_df[superstore_df["Order ID"].isin(multi_customer_orders.index)][
    ["Order ID", "Customer ID", "Customer Name", "Product ID", "Product Name"]
])

### Customer ID Inconsistency

Four Customer IDs (`HO-15231` to `HO-15234`) are associated with the same customer name, `Harry Olson`.

The same four IDs occur together across two separate orders. This suggests a possible duplicate or inconsistent customer identifier structure.

No correction will be applied without a defined business rule.

In [ ]:
profit_outlier=superstore_df[
    ((superstore_df["Profit"]-superstore_df["Profit"].mean())/superstore_df["Profit"].std()) > 3
][["Order ID","Customer ID","Product ID","Category","Profit"]]

print(f"Profit Outlier: \n {profit_outlier}")

## Data Quality Anomalies

In [ ]:
print(f"Discount Values: \n{superstore_df["Discount"].value_counts().sort_index()}")

print("Ship Modes: \n",superstore_df["Ship Mode"].value_counts())
print("Segment: \n",superstore_df["Segment"].value_counts())
print("Country/Region: \n",superstore_df["Country/Region"].value_counts())
print("Category: \n",superstore_df["Category"].value_counts())
print("Sub-Category: \n",superstore_df["Sub-Category"].value_counts())
shipping_days = (superstore_df["Ship Date"] - superstore_df["Order Date"]).dt.days

print(shipping_days.describe())


## Categorical Validation

- No obvious categorical inconsistencies were identified.
- Ship Mode, Segment, Country/Region, Category, and Sub-Category contain consistent values.


In [ ]:
print(f"Earliest Order Date :{superstore_df["Order Date"].min()}")
print(f"Latest Order Date :{superstore_df["Order Date"].max()}")
print(f"Earliest Ship Date :{superstore_df["Ship Date"].min()}")
print(f"Latest Ship Date :{superstore_df["Ship Date"].max()}")

ship_date_errors = (
    superstore_df["Ship Date"] < superstore_df["Order Date"]
).sum()

print(f"Ship Date errors: {ship_date_errors}")

print(f"Dataset Time span is: {((superstore_df["Order Date"].max())-(superstore_df["Order Date"].min())).days} days")


sales_error=(
    superstore_df["Sales"]<0
    ).sum()

print(f"Sales less than Zero : {sales_error}")

## Date Validation

- Ship Date is never earlier than Order Date.
- Shipping duration ranges from 0 to 11 days.
- 0-day shipping is considered valid because Same Day shipping exists.

In [ ]:
print(superstore_df["Sales"].describe())
print(superstore_df["Quantity"].describe())
print(superstore_df["Discount"].describe())
print(superstore_df["Profit"].describe())

## Numerical Validation

- Sales contains no negative values.
- Quantity contains positive values.
- Discount values fall within the observed range of 0 to 0.8.
- Profit contains both positive and negative values.
- Potential profit outliers were identified, but there is no evidence that they are invalid records.

In [ ]:

superstore_df["Postal Code"]=superstore_df["Postal Code"].astype(str)

print(f"Columns Datatype: \n{superstore_df.dtypes}")

## Data Types

- Date columns are stored as datetime.
- Numerical fields use appropriate numerical types.
- Postal Code is stored as a string because it is an identifier rather than a measure.

In [ ]:
missing_audit = []

for column in superstore_df.columns:
    missing_audit.append({
        "column_name": column,
        "missing_values": superstore_df[column].isna().sum(),
        "missing_%": superstore_df[column].isna().mean() * 100
    })

audit_df  = pd.DataFrame(missing_audit)


print(audit_df [["column_name", "missing_values", "missing_%"]])
print(f"duplicate Rows: {superstore_df.duplicated().sum()}")


## Data Completeness

- Missing values: None.
- Exact duplicate rows: None.

In [ ]:
import pandas as pd
from pathlib import Path


PROJECT_ROOT = Path(__file__).resolve().parents[2]


df= pd.read_excel(f"{PROJECT_ROOT}/data/raw/sample-superstore.xls")
superstore_df=pd.DataFrame(df)
print(f"Dataset: sample super store")
print(f"Columns: {len(superstore_df.columns)}")
print(f"Rows: {len(superstore_df)}")
print("Columns :")
for column in superstore_df.columns:
    print(column)

# Data Audit Summary
## Data Completeness

- Missing values: None.
- Exact duplicate rows: None.

## Data Types

- Date columns are stored as datetime.
- Numerical fields use appropriate numerical types.
- Postal Code is stored as a string because it is an identifier rather than a measure.

## Date Validation

- Ship Date is never earlier than Order Date.
- Shipping duration ranges from 0 to 11 days.
- 0-day shipping is considered valid because Same Day shipping exists.

## Categorical Validation

- No obvious categorical inconsistencies were identified.
- Ship Mode, Segment, Country/Region, Category, and Sub-Category contain consistent values.

## Numerical Validation

- Sales contains no negative values.
- Quantity contains positive values.
- Discount values fall within the observed range of 0 to 0.8.
- Profit contains both positive and negative values.
- Potential profit outliers were identified, but there is no evidence that they are invalid records.

## Data Quality Anomalies

### Customer ID Inconsistency

Four Customer IDs (`HO-15231` to `HO-15234`) are associated with the same customer name, `Harry Olson`.

The same four IDs occur together across two separate orders. This suggests a possible duplicate or inconsistent customer identifier structure.

No correction will be applied without a defined business rule.

### Repeated Order Composition

Two separate orders contain the same four products with identical quantities, discounts, sales, and profits.

Because the orders occurred on different dates, this is treated as a notable pattern rather than a confirmed data-quality error.

# Cleaning Plan

## Confirmed Cleaning Actions

1. Standardize column names for consistent Python and SQL usage.
2. Ensure date columns use the correct datetime format.
3. Remove leading and trailing whitespace from text fields if present.
4. Standardize categorical text formatting if inconsistencies are detected.
5. Validate numerical columns and enforce appropriate data types.
6. Create derived analytical fields such as Shipping Days where appropriate.

## Issues Requiring Review

1. Customer ID inconsistency for Harry Olson.
   - Do not arbitrarily merge or replace IDs.
   - Preserve the original identifiers.
   - Flag the records for review.

2. Potential profit outliers.
   - Retain the records.
   - Do not remove them because no evidence indicates that they are data-entry errors.

## Issues Requiring No Action

- Missing values: none.
- Exact duplicate rows: none.
- Negative sales: none.
- Invalid quantities: none.
- Ship dates earlier than order dates: none.
- Repeated order composition: retained because it may represent legitimate repeat purchases.